In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

In [78]:
def calculate_error_score_per_predicted_hour(df,forecast_prob_col,actual_is_peak_col):
    '''
    Score = (p-a)^2

    p is the forecast probability
    a is the actual outcome (0 if not the peak hour, 1 if it’s the daily peak hour)
    S is the score for that hour
    '''
    df = df.copy()
    df['error_score'] = (df[forecast_prob_col] - df[actual_is_peak_col]) ** 2

    return df

def calculate_softmax(x, temperature=2):
    x = np.array(x)
    x = x / temperature
    exp_x = np.exp(x - np.max(x))  # subtract max for numerical stability
    return exp_x / np.sum(exp_x)


def convert_day_preds_to_softmax(df, forecast_prob_col):
    df = df.copy()
    if 'datetime' not in df.columns:
        df['datetime'] = pd.to_datetime(df.index)
    # Apply softmax to the forecast probabilities for each day
    df['softmax_forecast_prob'] = df.groupby(df['datetime'].dt.date)[forecast_prob_col].transform(
        lambda x: calculate_softmax(x)
    )
    df['softmax_forecast_prob'] = df['softmax_forecast_prob'].astype(float).round(3)
    return df

def make_highest_daily_load_feature(df):
    df = df.copy()
    # extract date if not already separate
    
    
    # Initialize all as 0
    df['is_peak_hour_per_day'] = 0
    
    # For each day, find the index of the maximum load and mark it as 1
    for date in df['Date'].unique():
        daily_data = df[df['Date'] == date]
        peak_idx = daily_data['Load'].idxmax()
        df.loc[peak_idx, 'is_peak_hour_per_day'] = 1
    
    return df

def calculate_abs_error(df, actual_load, predicted_load):
    '''
    '''
    df = df.copy()
    df['rmse'] = abs(df[actual_load] - df[predicted_load])
    
    return df

def calculate_total_rmse(df, actual_load, predicted_load):
    '''
    RMSE = sqrt(mean((predicted - actual)^2))
    '''
    df = df.copy()
    df['squared_error'] = (df[actual_load] - df[predicted_load]) ** 2
    rmse = np.sqrt(df['squared_error'].mean())
    
    return rmse

def calculate_total_error(df, error_col):
    '''
    Total Error = sum of error scores for all 24 hours in the day
    '''
    total_error = df[error_col].sum()
    return total_error

In [82]:
pd.set_option('display.float_format', '{:.5f}'.format)
pred_path = '/home/leon/Documents/github_repos/amperon_take_home_test/Amperon_take_home/data/predictions/random_forest_val_set_preds.csv'
val_preds_df = pd.read_csv(pred_path)
# # if 'datetime' not in val_preds_df.columns:
# val_preds_df['datetime'] = pd.to_datetime(val_preds_df.index)
val_preds_df['datetime'] = pd.to_datetime(val_preds_df['datetime'])
val_preds_df['Date'] = val_preds_df['datetime'].dt.date
print(val_preds_df.head(24))

              datetime       Load      preds        Date
0  2007-01-01 01:00:00  741.00000  655.99000  2007-01-01
1  2007-01-01 02:00:00  678.00000  617.07000  2007-01-01
2  2007-01-01 03:00:00  626.00000  598.82000  2007-01-01
3  2007-01-01 04:00:00  603.00000  597.08000  2007-01-01
4  2007-01-01 05:00:00  598.00000  597.66000  2007-01-01
5  2007-01-01 06:00:00  612.00000  699.80000  2007-01-01
6  2007-01-01 07:00:00  638.00000  923.95000  2007-01-01
7  2007-01-01 08:00:00  678.00000  945.52000  2007-01-01
8  2007-01-01 09:00:00  744.00000  920.53000  2007-01-01
9  2007-01-01 10:00:00  851.00000  920.79000  2007-01-01
10 2007-01-01 11:00:00  943.00000  926.76000  2007-01-01
11 2007-01-01 12:00:00  989.00000  914.88000  2007-01-01
12 2007-01-01 13:00:00  966.00000  902.63000  2007-01-01
13 2007-01-01 14:00:00  933.00000  897.95000  2007-01-01
14 2007-01-01 15:00:00  920.00000  891.08000  2007-01-01
15 2007-01-01 16:00:00  912.00000  884.51000  2007-01-01
16 2007-01-01 17:00:00  918.000

In [85]:
val_preds_df_with_features = make_highest_daily_load_feature(val_preds_df)
val_preds_df_with_features = convert_day_preds_to_softmax(val_preds_df_with_features, 'preds')
val_preds_df_with_features = calculate_error_score_per_predicted_hour(val_preds_df_with_features, 'softmax_forecast_prob', 'is_peak_hour_per_day')
val_preds_df_with_features = calculate_abs_error(val_preds_df_with_features, 'Load', 'preds')
total_rmse = calculate_total_rmse(val_preds_df_with_features, 'Load', 'preds')
total_error = calculate_total_error(val_preds_df_with_features, 'error_score')
print('Total RMSE:', total_rmse)
print('Total Error:', total_error)

Total RMSE: 75.43013948198485
Total Error: 384.92263599999995


In [49]:
print(val_preds.groupby('datetime')['Load'])